# Notebook06 bounded training definitions

Reusable definitions only. Inputs: pinned Food101 training/development bundle, a frozen variant plan, and the original three-hour deadline. Outputs: immutable initialization states, full-state checkpoints and fixed-update evidence. Importing this notebook does not train. Training and probes are launched from `notebooks/06_Bounded_Mathematical_Improvements.ipynb`, inside detached tmux. Checkpoints live in `checkpoints/<run_id>/`; restart with Run All in a fresh `ahr-go` kernel using the unchanged run ID and source files. The original deadline never resets.

In [ ]:
"""Notebook06 finite-budget variant training; importing definitions never trains."""
from pathlib import Path
import copy, gc, hashlib, json, os, sys, time, types
import torch
from fmm_jepa.training import atomic_json, digest_file, definition_hashes
from fmm_jepa.food101_training import isolated_rng, _hash
import fmm_jepa.semantic_pilot as _pilot

ROOT = Path(__file__).resolve().parents[2]
RESUME_NOTEBOOK = 'notebooks/06_Bounded_Mathematical_Improvements.ipynb'
RESUME_CELL = 'launch-math-study'

def load_tagged(root, relative, name):
    """Load only the explicit reusable definition cells, once per process."""
    path=Path(root)/relative
    if name in sys.modules:
        module=sys.modules[name]
        if Path(module.__file__).resolve()!=path.resolve():raise ValueError('Module source path changed')
        return module
    module=types.ModuleType(name);module.__file__=str(path.resolve());sys.modules[name]=module
    cells=json.loads(path.read_text())['cells']
    selected=[c for c in cells if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
    if not selected:raise ValueError('Notebook has no reusable definition cells')
    try:
        for index,cell in enumerate(selected):
            exec(compile(''.join(cell['source']),str(path)+'#ahr-export-'+str(index),'exec'),module.__dict__)
    except BaseException:
        sys.modules.pop(name,None);raise
    return module

_BASE=load_tagged(ROOT,'experiments/05_scaling/runner.ipynb','fmm06_inherited_runner')

# Inherited functions resolve within an isolated module namespace. Upstream
# notebook files and canonical Notebook05 modules remain unchanged.
normalize_stage_plan=_BASE.normalize_stage_plan

def stage_definition_hashes(root):
    result=_BASE_ORIGINAL_HASHES(root)
    for path in sorted((Path(root)/'experiments/06_math').glob('*.ipynb')):
        cells=json.loads(path.read_text())['cells']
        sources=[''.join(c['source']) for c in cells if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
        result[str(path.relative_to(root))]=hashlib.sha256('\n'.join(sources).encode()).hexdigest()
    return result

_BASE_ORIGINAL_HASHES=_BASE.stage_definition_hashes

def build_model(config,kind):
    model=_pilot.build_model(config,kind)
    if kind=='semantic':
        options=config.get('math06')
        if not options:raise ValueError('Explicit math06 variant configuration required')
        backend=load_tagged(ROOT,'experiments/06_math/attention.ipynb','fmm06_attention')
        variant=options['variant'] if options['variant']!='vit' else 'stable'
        backend.configure_model(model,variant,options.get('feature_counts'))
    return model

def initialize_models(config,artifact):
    # Reuse the verified independent storage and common scaffold initialization.
    # Bind the extra operator configuration into its immutable identity.
    namespace=dict(_pilot.initialize_models.__globals__)
    namespace['build_model']=build_model
    namespace['_hash']=lambda value:_hash({'initialization':value,'math06':config.get('math06')})
    function=types.FunctionType(_pilot.initialize_models.__code__,namespace,
        _pilot.initialize_models.__name__,_pilot.initialize_models.__defaults__,_pilot.initialize_models.__closure__)
    return function(config,artifact)

for _name in ('_strict_training_protocol',):
    globals()[_name]=getattr(_BASE,_name)

def _plan_binding(root,plan,bundle):
    safe=lambda s:isinstance(s,str) and bool(s) and all(c.isalnum() or c in '-_' for c in s)
    if not safe(plan.get('run_id')) or not safe(plan.get('stage_id')):raise ValueError('Safe run_id and stage_id required')
    arms=plan.get('arms',[])
    if not arms or len({a['id'] for a in arms})!=len(arms) or any(not safe(a['id']) for a in arms):
        raise ValueError('Unique safe arm IDs required')
    updates=int(plan['updates']);milestones=list(map(int,plan.get('milestones',[300,updates])))
    if updates<4 or len(set(milestones))!=len(milestones) or any(m<1 or m>updates for m in milestones):
        raise ValueError('Invalid schedule or milestone set')
    if updates not in milestones:raise ValueError('Final update must be an explicit milestone')
    if int(plan.get('round_updates',100))<1:raise ValueError('Positive round size required')
    train=list(bundle['train_ids']);probe=list(bundle['probe_train_ids']);dev=list(bundle['dev_ids']);heldout=list(bundle.get('heldout_ids',[]))
    if not train or not probe or not dev or any(len(ids)!=len(set(ids)) for ids in (train,probe,dev)):
        raise ValueError('Nonempty unique training/probe/development memberships required')
    if set(train)&set(dev) or set(probe)&set(dev) or (set(train)|set(probe)|set(dev))&set(heldout):
        raise ValueError('Development or official-test leakage')
    if not set(probe).issubset(train):raise ValueError('Fixed probe training set must be nested inside stage pretraining pool')
    if 'train_count' in plan and int(plan['train_count'])!=len(train):raise ValueError('Stage training count differs from plan')
    if 'probe_train_count' in plan and int(plan['probe_train_count'])!=len(probe):raise ValueError('Probe count differs from plan')
    seeds=sorted({int(a['seed']) for a in arms})
    if 'seeds' in plan and sorted(map(int,plan['seeds']))!=seeds:raise ValueError('Declared seed list differs from arms')
    for arm in arms:
        config=arm['config'];policy=_strict_training_protocol(config)
        if arm['kind'] not in ('vit','semantic') or int(config['seeds']['model_init'])!=int(arm['seed']):
            raise ValueError('Architecture or model seed mismatch')
        if policy['updates']!=updates or policy['effective_batch']!=int(plan.get('effective_batch',128)):
            raise ValueError('Schedule/effective batch differs from registered stage')
        if config.get('dtype','float32')!='float32':raise ValueError('Only verified FP32 enabled')
        if config.get('evaluation',{}).get('heldout_enabled',False):raise ValueError('Official test is reserved')
        if int(config['probe']['epochs'])!=20:raise ValueError('Fixed epoch20 probes required')
        if policy['checkpoint_every']>50:raise ValueError('Checkpoint at most every50 successful updates')
    variants=plan.get('variants',['vit','stable','centered_tree'])
    if len(seeds) not in (2,3):raise ValueError('Exactly two or three paired training seeds required')
    if len(set(variants))!=len(variants) or not {'vit','stable','centered_tree'}.issubset(set(variants)):
        raise ValueError('Predeclared ViT, stable, centered_tree controls are required')
    for seed in seeds:
        group=[a for a in arms if int(a['seed'])==seed]
        if sorted(a['variant'] for a in group)!=sorted(variants):raise ValueError('Every seed must contain every registered variant exactly once')
        configs=[]
        for arm in group:
            config=normalize_stage_plan(arm['config'])
            options=config.pop('math06',None)
            if arm['variant']=='vit':
                if arm['kind']!='vit':raise ValueError('ViT variant has wrong architecture')
            elif arm['kind']!='semantic' or not options or options['variant']!=arm['variant']:
                raise ValueError('Semantic variant/config mismatch')
            config.get('model',{}).pop('kind',None)
            configs.append(config)
        if any(config!=configs[0] for config in configs[1:]):
            raise ValueError('Paired variants differ beyond architecture and registered math06 operator options')
    sources=plan.get('source_hashes',stage_definition_hashes(root));actual=stage_definition_hashes(root)
    if sources!=actual:raise ValueError('Registered Notebook05 definition hashes changed')
    consumer_hash=plan.get('consumer_notebook_sha256')
    if consumer_hash:
        nb=json.loads((Path(root)/plan.get('resume_notebook',RESUME_NOTEBOOK)).read_text())
        code=[''.join(c['source']) for c in nb['cells'] if c['cell_type']=='code']
        if _hash(code)!=consumer_hash:raise ValueError('Registered main notebook code changed')
    return {'schema':'fmm-jepa-notebook06-stage-binding/v1','plan_hash':_hash(plan),
        'source_hashes':sources,'dataset_fingerprint':bundle['dataset'].fingerprint,
        'split_sha256':bundle['split_manifest']['split_sha256'],
        'training_ids_sha256':_hash(train),'training_examples':len(train),
        'probe_training_ids_sha256':_hash(probe),'probe_training_examples':len(probe),
        'development_ids_sha256':_hash(dev),'development_examples':len(dev),'heldout_status':'NOT_RUN'}

def _persist_numerical_backend_stats(artifact,stage_id,arm_id,event):
    backend=sys.modules.get('fmm06_attention')
    value={'status':'RECORDED','stage_id':stage_id,'arm_id':arm_id,'event':event,
        'updated_unix':time.time(),'worker_pid':os.getpid(),
        'counter_scope':'Current process cumulative, including calibration/preflight and all earlier arms; not checkpoint state or training-only performance.'}
    for name in ('get_stats','stats'):
        if backend is not None and callable(getattr(backend,name,None)):
            value['statistics']=getattr(backend,name)();break
    atomic_json(Path(artifact)/'numerical_backend_stats.json',value)
    return value

for _name in ('build_model','initialize_models','stage_definition_hashes','_plan_binding','_persist_numerical_backend_stats'):
    setattr(_BASE,_name,globals()[_name])
_BASE.RESUME_NOTEBOOK=RESUME_NOTEBOOK
_BASE.RESUME_CELL=RESUME_CELL
# Make the inherited globals available to the deliberately adapted preflight.
for _name in ('_rng','_restore_rng','_atomic_torch','_record','_guard','_deadline','_status',
              '_registered','_arm_paths','_initial_model','_run_stage_model','_release_semantic','validate_training_manifest'):
    globals()[_name]=getattr(_BASE,_name)
def verify_stage_resume(root,artifact,plan,bundle,deadline_unix=None):
    """At user execution only: actual-config four updates equals reconstructed2+2.

    One first-seed certificate per architecture covers that stage resolution,
    batch/accumulation, data membership and full successful-update schedule.
    Every RNG/optimizer/model/scheduler/EMA/sampler/trace is compared bitwise;
    elapsed timing is the only excluded state.
    """
    from fmm_jepa.food101_benchmark import _food_exact_tree
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    registration=_registered(root,artifact,plan,bundle);path=artifact/'stage_preflight.json'
    if path.exists():
        certificate=json.loads(path.read_text())
        if certificate['status']!='PASS' or certificate['binding']!=registration['binding']:
            raise ValueError('Actual-config preflight identity differs')
        expected={next(a['id'] for a in plan['arms'] if a['variant']==variant) for variant in plan['variants']}
        if set(certificate['arms'])!=expected:raise ValueError('Resume certificate must cover every registered variant')
        for entry in certificate['arms'].values():
            for record in entry['checkpoints'].values():
                folder=Path(record['directory']);manifest=validate_training_manifest(folder)
                if manifest['successful_updates']!=4 or digest_file(folder/manifest['latest'])!=record['sha256']:
                    raise ValueError('Full-state preflight evidence changed')
        return certificate
    selected=[]
    for variant in plan['variants']:
        selected.append(next(a for a in plan['arms'] if a['variant']==variant))
    fields=['model','optimizer','scheduler','ema_position','scaler','precision','rng','sampler',
            'streams','microstep','partial_accumulation','world_size','rank','replay','config','protocol','identity']
    state_fields=['successful_updates','attempted_updates','image_exposures','consecutive_failures','traces','failures','diagnostics']
    timing={'data_seconds','compute_seconds','step_seconds','images_per_second','_cuda_event_count'}
    certificate={'status':'RUNNING','binding':registration['binding'],'arms':{},
        'scope':'First seed per registered variant; exact actual stage config/data and full schedule; elapsed timing excluded'}
    original_rng=_rng()
    try:
        with _guard(deadline_unix):
            for arm in selected:
                _deadline(deadline_unix,5);_status(artifact,'exact_resume','RUNNING',arm_id=arm['id'])
                arm_folder,_=_arm_paths(root,artifact,plan,arm)
                protocol=json.loads((arm_folder/'protocol.json').read_text())
                folder=artifact/'preflight'/arm['id'];folder.mkdir(parents=True,exist_ok=True)
                rng_path=folder/'trajectory_rng.pt';rng_record=folder/'trajectory_rng.json'
                if rng_record.exists():
                    if digest_file(rng_path)!=json.loads(rng_record.read_text())['sha256']:raise ValueError('Preflight RNG changed')
                    trajectory=torch.load(rng_path,map_location='cpu',weights_only=False)
                else:
                    if rng_path.exists():raise ValueError('Orphan trajectory RNG; inspect before recovery')
                    with isolated_rng(int(arm['seed'])):trajectory=_rng()
                    _atomic_torch(rng_path,trajectory);atomic_json(rng_record,_record(rng_path))
                reference_dir=root/'checkpoints'/plan['run_id']/'stages'/plan['stage_id']/'preflight'/(arm['id']+'-reference')
                split_dir=reference_dir.with_name(arm['id']+'-split')
                _restore_rng(copy.deepcopy(trajectory))
                reference=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'reference',reference_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                del reference
                _restore_rng(copy.deepcopy(trajectory))
                partial=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=2,deadline_unix=deadline_unix)
                if partial['resume']['final_step'] not in (2,4):raise AssertionError('Preflight cutoff before split boundary')
                del partial
                resumed=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                lm,rm=validate_training_manifest(reference_dir),validate_training_manifest(split_dir)
                left=torch.load(reference_dir/lm['latest'],map_location='cpu',weights_only=False)
                right=torch.load(split_dir/rm['latest'],map_location='cpu',weights_only=False)
                for field in fields:_food_exact_tree(left[field],right[field],field)
                for field in state_fields:_food_exact_tree(left['state'][field],right['state'][field],'state.'+field)
                lmetrics=[{k:v for k,v in row.items() if k not in timing} for row in left['state']['metrics']]
                rmetrics=[{k:v for k,v in row.items() if k not in timing} for row in right['state']['metrics']]
                _food_exact_tree(lmetrics,rmetrics,'deterministic_metrics')
                if lm['successful_updates']!=4 or rm['attempted_updates']!=4 or len(rm['checkpoints'])<2:
                    raise AssertionError('Four finite updates and latest-two retention required')
                certificate['arms'][arm['id']]={'status':'PASS','bitwise_equal':True,
                    'image_size':arm['config']['image_size'],'training_examples':len(bundle['train_ids']),
                    'microbatch':arm['config']['training']['microbatch'],'accumulation_steps':arm['config']['training']['accumulation_steps'],
                    'schedule_updates':plan['updates'],'full_state_fields':fields,
                    'checkpoints':{'reference':{'directory':str(reference_dir),'sha256':digest_file(reference_dir/lm['latest'])},
                                   'resumed':{'directory':str(split_dir),'sha256':digest_file(split_dir/rm['latest'])}}}
                atomic_json(folder/'certificate.json',certificate['arms'][arm['id']])
                del left,right,resumed;_release_semantic(arm['config']['device'])
    finally:_restore_rng(original_rng)
    certificate['status']='PASS';atomic_json(path,certificate)
    return certificate
_BASE.verify_stage_resume=verify_stage_resume
prepare_stage=_BASE.prepare_stage
train_stage=_BASE.train_stage
validate_stage_milestones=_BASE.validate_stage_milestones

def calibrate_case(root,artifact,plan,arm,bundle,stop_after=8,deadline_unix=None):
    """Discarded actual full-update calibration; never supplies a quality score."""
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    models,initial=initialize_models(arm['config'],artifact)
    model=models[arm['kind']]
    del models
    protocol={'schema':'math06-discarded-calibration/v1','arm':arm,
        'source_hashes':stage_definition_hashes(root),'initialization':initial,
        'stop_after':int(stop_after),'quality_selection':'FORBIDDEN','heldout_status':'NOT_RUN'}
    checkpoint=Path(root)/'checkpoints'/plan['run_id']/'calibration'/arm['id']
    result=_run_stage_model(plan,arm['kind'],arm['config'],bundle,artifact,checkpoint,protocol,
        model=model,stop_after=int(stop_after),deadline_unix=deadline_unix)
    if result['resume']['final_step']!=int(stop_after) or any(not r.get('successful',False) for r in result['traces']):
        raise RuntimeError('Calibration must complete all finite updates')
    _release_semantic(arm['config']['device'])
    return result
